In [ ]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# Load the dataset
file_path = 'C:/Users/nosao/Desktop/Maxwell-Text Classification/Target Response/data/target_work_environment.csv' #Replace with actual file path
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()  # Clean column names

# Clean missing values
df = df.dropna(subset=['Job description', 'Question 37', 'Question 38', 'Question 39'])

# Extract job descriptions and target variables
X = df['Job description'].astype(str)
y_q37 = df['Question 37']
y_q38 = df['Question 38']
y_q39 = df['Question 39']

# Split data into training and test sets for each question
X_train, X_test, y_train_37, y_test_37 = train_test_split(X, y_q37, test_size=0.2, random_state=42)
X_train_q38, X_test_q38, y_train_q38, y_test_q38 = train_test_split(X, y_q38, test_size=0.2, random_state=42)
X_train_q39, X_test_q39, y_train_q39, y_test_q39 = train_test_split(X, y_q39, test_size=0.2, random_state=42)

# Define Logistic Regression with class_weight='balanced' to handle class imbalance
logreg_pipeline = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, class_weight='balanced'))

# Hyperparameter tuning grid
param_grid_logreg = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'tfidfvectorizer__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidfvectorizer__max_df': [0.85, 0.9, 0.95],
    'tfidfvectorizer__min_df': [1, 5],
    'tfidfvectorizer__use_idf': [True, False],
    'tfidfvectorizer__sublinear_tf': [True, False]
}

# Hyperparameter tuning with StratifiedKFold to ensure balanced class representation in cross-validation
cv = StratifiedKFold(n_splits=5)

# Hyperparameter tuning for Logistic Regression for each question
grid_logreg_q37 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q37.fit(X_train, y_train_37)

grid_logreg_q38 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q38.fit(X_train_q38, y_train_q38)

grid_logreg_q39 = GridSearchCV(logreg_pipeline, param_grid_logreg, cv=cv, scoring='accuracy', n_jobs=-1)
grid_logreg_q39.fit(X_train_q39, y_train_q39)

# Use the best Logistic Regression models for each question
best_logreg_model_q37 = grid_logreg_q37.best_estimator_
best_logreg_model_q38 = grid_logreg_q38.best_estimator_
best_logreg_model_q39 = grid_logreg_q39.best_estimator_

# Function to display metrics
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy: {accuracy_score(y_true, y_pred)}\n")

# Predict on the full test set for each question and display metrics
y_pred_q37 = best_logreg_model_q37.predict(X_test)
display_metrics(y_test_37, y_pred_q37, 37)

y_pred_q38 = best_logreg_model_q38.predict(X_test_q38)
display_metrics(y_test_q38, y_pred_q38, 38)

y_pred_q39 = best_logreg_model_q39.predict(X_test_q39)
display_metrics(y_test_q39, y_pred_q39, 39)

# Function to make predictions on a new job description
def predict_for_new_job_description(job_description):
    # Ensure the input is a string
    job_description = [job_description]

    # Predict for each question using the best model
    pred_q37 = best_logreg_model_q37.predict(job_description)[0]
    pred_q38 = best_logreg_model_q38.predict(job_description)[0]
    pred_q39 = best_logreg_model_q39.predict(job_description)[0]

    # Display or return the results
    print("Predictions for the new job description:")
    print(f"Question 37: {pred_q37}")
    print(f"Question 38: {pred_q38}")
    print(f"Question 39: {pred_q39}")

# Example: Enter a new job description
new_job_description = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.

"""

# Function to enforce rule-based evaluation after predictions
def apply_rule_based_correction(predictions):
    """
    Apply rule-based correction to predictions.
    Ensures at least one question gets 'A' and updates preceding questions to 'D'.
    """
    # Question order
    questions = [37, 38, 39]
    
    # If 'A' is already in predictions, apply the rule
    if 'A' in predictions.values():
        a_index = list(predictions.values()).index('A')  # Find the first occurrence of 'A'
        
        # Set all preceding questions to 'C'
        for i in range(a_index):
            predictions[questions[i]] = 'C'
    
    else:
        # If no 'A' is assigned, select the best candidate
        for q in questions:
            if predictions[q] in ['B']:  
                predictions[q] = 'A'
                a_index = questions.index(q)
                
                # Set all previous questions to 'C'
                for i in range(a_index):
                    predictions[questions[i]] = 'C'
                break  # Ensure only one 'A' is assigned

    return predictions

# Function to make predictions on a new job description with rule enforcement
def predict_for_new_job_description(job_description):
    job_description = [job_description]  # Ensure it's a list for model input
    
    # Model predictions
    predictions = {
        37: best_logreg_model_q37.predict(job_description)[0],
        38: best_logreg_model_q38.predict(job_description)[0],
        39: best_logreg_model_q39.predict(job_description)[0],
    }
    
    # Apply rule-based correction
    corrected_predictions = apply_rule_based_correction(predictions)

    # Display results
    print("Corrected Predictions for the new job description:")
    for q, pred in corrected_predictions.items():
        print(f"Question {q}: {pred}")


# Call the function with the job description
predict_for_new_job_description(new_job_description)

# Create a folder for saving models
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)  # Create the folder 

# Save the models in the "saved_models" directory
joblib.dump(best_logreg_model_q37, os.path.join(save_dir, "model_q37.pkl"))
joblib.dump(best_logreg_model_q38, os.path.join(save_dir, "model_q38.pkl"))
joblib.dump(best_logreg_model_q39, os.path.join(save_dir, "model_q39.pkl"))

print("Models saved in the 'saved_models' folder.")

c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\nosao\Desktop\spacyProj\spacy_venv\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Metrics for Question 37
              precision    recall  f1-score   support

           C       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19

Accuracy: 1.0

Metrics for Question 38
              precision    recall  f1-score   support

           B       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19

Accuracy: 1.0

Metrics for Question 39
              precision    recall  f1-score   support

           A       1.00      1.00      1.00        19

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19

Accuracy: 1.0

Corrected Predictions for the new job description:
Question 37: 

In [3]:
import joblib
import os

# Define the saved models directory
save_dir = "saved_models"

# Load models correctly
best_logreg_model_q37 = joblib.load(os.path.join(save_dir, "model_q37.pkl"))
best_logreg_model_q38 = joblib.load(os.path.join(save_dir, "model_q38.pkl"))
best_logreg_model_q39 = joblib.load(os.path.join(save_dir, "model_q39.pkl"))

print("Models loaded successfully!")

Models loaded successfully!


In [4]:
# Example: Enter a new job description
new_job_description = """



"""

# Function to enforce rule-based evaluation after predictions
def apply_rule_based_correction(predictions):
    """
    Apply rule-based correction to predictions.
    Ensures at least one question gets 'A' and updates preceding questions to 'D'.
    """
    # Question order
    questions = [37, 38, 39]
    
    # If 'A' is already in predictions, apply the rule
    if 'A' in predictions.values():
        a_index = list(predictions.values()).index('A')  # Find the first occurrence of 'A'
        
        # Set all preceding questions to 'C'
        for i in range(a_index):
            predictions[questions[i]] = 'C'
    
    else:
        # If no 'A' is assigned, select the best candidate
        for q in questions:
            if predictions[q] in ['B']:  # Prefer B over C
                predictions[q] = 'A'
                a_index = questions.index(q)
                
                # Set all previous questions to 'C'
                for i in range(a_index):
                    predictions[questions[i]] = 'C'
                break  # Ensure only one 'A' is assigned

    return predictions

# Function to make predictions on a new job description with rule enforcement
def predict_for_new_job_description(job_description):
    job_description = [job_description]  # Ensure it's a list for model input
    
    # Model predictions
    predictions = {
        37: best_logreg_model_q37.predict(job_description)[0],
        38: best_logreg_model_q38.predict(job_description)[0],
        39: best_logreg_model_q39.predict(job_description)[0],
    }
    
    # Apply rule-based correction
    corrected_predictions = apply_rule_based_correction(predictions)

    # Display results
    print("Corrected Predictions for the new job description:")
    for q, pred in corrected_predictions.items():
        print(f"Question {q}: {pred}")


# Call the function with the job description
predict_for_new_job_description(new_job_description)

Corrected Predictions for the new job description:
Question 37: C
Question 38: C
Question 39: A
